# Simple benchmark study sheet

Generates `docs/simple-benchmark-mc-study-sheet.txt`: every `mc_full_probs` prompt grouped by `vignette_name`, with P(C), P(D), P(T|C), P(T|D) for the well-posed case and both implausible forks, plus **score_pct** per variant (`open_probs`, `mc_numeric_probs`, `mc_full_probs`) pooled across all downloaded Kaggle models.

Reads `data/simple/benchmark.csv`. Scores come from `data/kaggle_runs/simple-rate-normative-accuracy/` when present (download via `simple-results.ipynb` or `python scripts/export_simple_rate_kaggle_results.py --download`).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "data" / "simple").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repo root:", ROOT)

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

from benchmarks.kaggle_runs import (
    DEFAULT_SIMPLE_RATE_TASK_SLUG,
    merged_simple_results_from_kaggle_runs,
)

WIDTH = 80
SEP = "=" * WIDTH
SUB = "-" * WIDTH
VARIANT_ORDER = ("open_probs", "mc_numeric_probs", "mc_full_probs")

PROBLEM_TYPE_ORDER = {"well_posed": 0, "implausible_c_d": 1, "implausible_t": 2}
PROBLEM_TYPE_LABEL = {
    "well_posed": "well-posed",
    "implausible_c_d": "implausible P(C)/P(D)",
    "implausible_t": "implausible P(T|·)",
}


def _simple_pct(value: float) -> str:
    if value < 0.01:
        return f"{value * 100:.2f}%"
    rounded = round(value * 100)
    if abs(value * 100 - rounded) <= 0.05:
        return f"{rounded}%"
    return f"{value * 100:.1f}%"


def _prob_line(row: dict[str, str]) -> str:
    return (
        f"P(C)={_simple_pct(float(row['p_c']))}  "
        f"P(D)={_simple_pct(float(row['p_d']))}  "
        f"P(T|C)={_simple_pct(float(row['p_t_given_c']))}  "
        f"P(T|D)={_simple_pct(float(row['p_t_given_d']))}"
    )


def _score_value(row: dict[str, str]) -> int:
    return 1 if str(row.get("score", "")).lower() == "true" else 0


def _load_vignette_variant_scores(root: Path) -> dict[str, dict[str, tuple[float | None, int]]]:
    kaggle_dir = root / "data" / "kaggle_runs" / DEFAULT_SIMPLE_RATE_TASK_SLUG
    benchmark_csv = root / "data" / "simple" / "benchmark.csv"
    if not kaggle_dir.is_dir():
        return {}

    merged_rows = merged_simple_results_from_kaggle_runs(
        kaggle_dir,
        benchmark_path=benchmark_csv,
        fill_missing=False,
    )
    buckets: dict[str, dict[str, list[int]]] = defaultdict(lambda: defaultdict(list))
    for row in merged_rows:
        variant = (row.get("variant") or "").strip()
        if variant not in VARIANT_ORDER:
            continue
        buckets[row["vignette_name"]][variant].append(_score_value(row))

    scores: dict[str, dict[str, tuple[float | None, int]]] = {}
    for vignette, variants in buckets.items():
        scores[vignette] = {}
        for variant in VARIANT_ORDER:
            values = variants.get(variant, [])
            if values:
                scores[vignette][variant] = (
                    round(sum(values) / len(values) * 100, 1),
                    len(values),
                )
            else:
                scores[vignette][variant] = (None, 0)
    return scores


def _fmt_variant_score(pct: float | None, n: int) -> str:
    if pct is None or n == 0:
        return "—"
    return f"{pct:.1f}% (n={n})"


benchmark_csv = ROOT / "data" / "simple" / "benchmark.csv"
out_path = ROOT / "docs" / "simple-benchmark-mc-study-sheet.txt"
vignette_scores = _load_vignette_variant_scores(ROOT)

with benchmark_csv.open(newline="", encoding="utf-8") as handle:
    rows = [
        row
        for row in csv.DictReader(handle)
        if row["variant"] == "mc_full_probs"
    ]

by_vignette: dict[str, list[dict[str, str]]] = defaultdict(list)
for row in rows:
    by_vignette[row["vignette_name"]].append(row)
for vignette_rows in by_vignette.values():
    vignette_rows.sort(
        key=lambda row: (
            PROBLEM_TYPE_ORDER.get(row["problem_type"], 99),
            row["example_id"],
        )
    )

lines: list[str] = [
    SEP,
    "SIMPLE BENCHMARK — MC FULL STUDY SHEET",
    SEP,
    "",
    f"Source: {benchmark_csv.relative_to(ROOT)}",
    f"Rows: {len(rows)} mc_full_probs prompts across {len(by_vignette)} vignettes",
]
if vignette_scores:
    lines.append(
        f"Scores: pooled across models from data/kaggle_runs/{DEFAULT_SIMPLE_RATE_TASK_SLUG}/"
    )
else:
    lines.append(
        f"Scores: (no runs in data/kaggle_runs/{DEFAULT_SIMPLE_RATE_TASK_SLUG}/)"
    )
lines.append("")

for vignette in sorted(by_vignette, key=str.lower):
    vignette_rows = by_vignette[vignette]
    by_type = {row["problem_type"]: row for row in vignette_rows}

    lines.extend([SEP, vignette.upper(), SEP, ""])
    if vignette in vignette_scores:
        lines.append("Scores (mean normative pass rate, all models):")
        for variant in VARIANT_ORDER:
            pct, n = vignette_scores[vignette].get(variant, (None, 0))
            lines.append(
                f"  {variant:<18} {_fmt_variant_score(pct, n)}"
            )
        lines.append("")
    lines.extend(["Probabilities:"])
    for problem_type in ("well_posed", "implausible_c_d", "implausible_t"):
        row = by_type[problem_type]
        lines.append(f"  {PROBLEM_TYPE_LABEL[problem_type]:<22} {_prob_line(row)}")
    lines.append("")

    for row in vignette_rows:
        problem = row["problem_type"]
        label = PROBLEM_TYPE_LABEL.get(problem, problem)
        scepticism = row["scepticism_required"].strip().lower() == "true"
        score_target = row["scepticism_score_target"].strip()
        normative = f"{row['normative_choice']} ({row['normative_percent']})"

        lines.extend(
            [
                SUB,
                f"{label}  |  example_id: {row['example_id']}",
                f"normative: {normative}"
                + (f"  |  scepticism target: {score_target}" if scepticism else ""),
                f"probs: {_prob_line(row)}",
                SUB,
                "",
                row["prompt"].strip(),
                "",
            ]
        )
    lines.append("")

out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Wrote {len(rows)} prompts to {out_path}")
if vignette_scores:
    print(f"Included scores for {len(vignette_scores)} vignettes")
print(f"Size: {out_path.stat().st_size:,} bytes")